# Exploratory Data Analysis (EDA)


This notebook explores the merged dataset produced by the four preprocessing notebooks.
It covers:
1. Data loading & merging
2. Dataset overview & quality checks
3. Sales patterns (hourly, daily, monthly, yearly)
4. Top SKUs & category analysis
5. Weather correlations
6. Event & holiday impact
7. Outlier & anomaly detection
8. Key findings summary

## 0. Imports & Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Plotting style ──────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'axes.grid':        True,
    'grid.alpha':       0.3,
    'font.size':        11,
    'axes.titlesize':   13,
    'axes.titleweight': 'bold',
})
PALETTE = '#2E4057'
ACCENT  = '#E07A5F'
print('Setup complete.')

Setup complete.


# Data Ingestion & Merging
# ==========================================
# Load and consolidate processed data sources into a master dataframe.

In [2]:
# ── Load each processed source ───────────────────────────────────────────────
sales    = pd.read_csv('processed_csv/sales_data_preprocessed.csv',            parse_dates=['Date'])
weather  = pd.read_csv('processed_csv/weather_data_hourly.csv',                parse_dates=['Date'])
events   = pd.read_csv('processed_csv/aberdeen_events_master_timeline.csv',    parse_dates=['Date'])
holidays = pd.read_csv('processed_csv/holidays_data_preprocessed.csv',         parse_dates=['Date'])

print(f'Sales    : {sales.shape}')
print(f'Weather  : {weather.shape}')
print(f'Events   : {events.shape}')
print(f'Holidays : {holidays.shape}')

Sales    : (13149, 223)
Weather  : (13149, 11)
Events   : (13167, 5)
Holidays : (13149, 6)


In [ ]:
# ── Merge all sources on Date ────────────────────────────────────────────────
# sales, weather, and events have hourly timestamps.
# holidays only has daily dates.
df = sales.merge(weather,  on='Date', how='left')
df = df.merge(events,   on='Date', how='left')

# To merge holidays, we create a temporary date-only column
df['Date_only'] = df['Date'].dt.date
holidays['Date_only'] = holidays['Date'].dt.date
df = df.merge(holidays.drop(columns=['Date']), on='Date_only', how='left').drop(columns=['Date_only'])

# Fill NaN values for holiday and event flags (left merge may result in NaNs)
df['is_holiday'] = df['is_holiday'].fillna(0).astype(int)
df['holiday_importance'] = df['holiday_importance'].fillna(0)
df['is_festival'] = df['is_festival'].fillna(0).astype(int)
df['is_match_day'] = df['is_match_day'].fillna(0).astype(int)

# ── Define SKU columns (everything except Date and context columns) ──────────
context_cols = [
    'Date',
    # weather
    'apparent_temperature', 'precipitation', 'snowfall', 'snow_depth',
    'weather_code', 'relative_humidity_2m', 'cloud_cover',
    'visibility', 'wind_speed_10m', 'wind_gusts_10m',
    # events
    'is_festival', 'festival_importance', 'is_match_day', 'match_importance',
    # holidays
    'is_holiday', 'holiday_importance', 'Days_Until_Next_Holiday'
]
sku_cols = [c for c in df.columns if c not in context_cols]

# ── Total sales per row (useful for aggregate analysis) ──────────────────────
# Filter sku_cols to only include numeric columns to avoid object dtype issues
sku_cols = [c for c in sku_cols if pd.api.types.is_numeric_dtype(df[c])]
df['total_items_sold'] = df[sku_cols].sum(axis=1)

print(f'Master dataframe : {df.shape}')
print(f'SKU columns      : {len(sku_cols)}')
df.head(3)

# Data Quality Assessment
# ==========================================

In [ ]:
print('=== Date Range ===')
print(f"  From : {df['Date'].min()}")
print(f"  To   : {df['Date'].max()}")
print(f"  Rows : {len(df):,}")

print('\n=== Null Values (context columns only) ===')
nulls = df[context_cols].isnull().sum()
print(nulls[nulls > 0] if nulls.any() else 'No nulls found — merge successful.')

print('\n=== SKU columns with ALL zeros (never sold) ===')
never_sold = [c for c in sku_cols if df[c].sum() == 0]
print(f'  Count : {len(never_sold)}')
if never_sold:
    print(f'  Examples : {never_sold[:10]}')

In [ ]:
# ── Flag zero-sales hours (possible closure days) ────────────────────────────
zero_hours = df[df['total_items_sold'] == 0].copy()
zero_days  = zero_hours.groupby(zero_hours['Date'].dt.date).size()
full_closure_days = zero_days[zero_days == 9]   # all 9 business hours = 0

print(f'Hours with zero sales : {len(zero_hours):,}')
print(f'Days with ALL zero hours (likely closures) : {len(full_closure_days)}')
if len(full_closure_days) > 0:
    print('\nFirst 10 closure days:')
    print(full_closure_days.head(10))

# Sales Pattern Analysis
# ==========================================

### 3.1 Overall sales trend over 4 years

In [ ]:
import os
if not os.path.exists('eda_outputs'):
    os.makedirs('eda_outputs')

weekly_sales = df.resample('W', on='Date')['total_items_sold'].sum()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(weekly_sales.index, weekly_sales.values, color=PALETTE, linewidth=1.2)
ax.fill_between(weekly_sales.index, weekly_sales.values, alpha=0.15, color=PALETTE)
ax.set_title('Weekly Total Items Sold — 2022 to 2025')
ax.set_xlabel('Date')
ax.set_ylabel('Items Sold')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('eda_outputs/01_weekly_sales_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Peak week   : {weekly_sales.idxmax().date()} ({weekly_sales.max():,.0f} items)')
print(f'Lowest week : {weekly_sales.idxmin().date()} ({weekly_sales.min():,.0f} items)')

### 3.2 Average sales by hour of day

In [ ]:
# Exclude full closure days to avoid bias
closure_dates = set(full_closure_days.index)
df_open = df[~df['Date'].dt.date.isin(closure_dates)].copy()

hourly_avg = df_open.groupby(df_open['Date'].dt.hour)['total_items_sold'].mean()

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(hourly_avg.index, hourly_avg.values, color=PALETTE, width=0.6, edgecolor='white')
# Highlight the peak hour
peak_h = hourly_avg.idxmax()
bars[peak_h - 8].set_color(ACCENT)
ax.set_title('Average Items Sold by Hour of Day (open days only)')
ax.set_xlabel('Hour')
ax.set_ylabel('Avg Items Sold')
ax.set_xticks(range(8, 17))
ax.set_xticklabels([f'{h:02d}:00' for h in range(8, 17)])
plt.tight_layout()
plt.savefig('eda_outputs/02_hourly_avg_sales.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Peak hour : {peak_h:02d}:00 ({hourly_avg[peak_h]:.1f} avg items)')

### 3.3 Average sales by day of week

In [ ]:
day_labels = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# Daily totals first, then average by weekday
daily_totals = df_open.groupby(df_open['Date'].dt.date)['total_items_sold'].sum().reset_index()
daily_totals.columns = ['date', 'total']
daily_totals['weekday'] = pd.to_datetime(daily_totals['date']).dt.dayofweek
weekday_avg = daily_totals.groupby('weekday')['total'].mean()

fig, ax = plt.subplots(figsize=(9, 4))
colors = [ACCENT if i == weekday_avg.idxmax() else PALETTE for i in range(7)]
ax.bar(range(7), weekday_avg.values, color=colors, width=0.6, edgecolor='white')
ax.set_title('Average Daily Items Sold by Day of Week')
ax.set_xlabel('Day')
ax.set_ylabel('Avg Items Sold')
ax.set_xticks(range(7))
ax.set_xticklabels(day_labels, rotation=30, ha='right')
plt.tight_layout()
plt.savefig('eda_outputs/03_weekday_avg_sales.png', dpi=150, bbox_inches='tight')
plt.show()
print('Average sales by weekday:')
for i, v in weekday_avg.items():
    print(f'  {day_labels[i]:<12}: {v:.0f}')

### 3.4 Average sales by month (seasonality)

In [ ]:
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

daily_totals2 = df_open.groupby(df_open['Date'].dt.date)['total_items_sold'].sum().reset_index()
daily_totals2.columns = ['date', 'total']
daily_totals2['month'] = pd.to_datetime(daily_totals2['date']).dt.month
monthly_avg = daily_totals2.groupby('month')['total'].mean()

fig, ax = plt.subplots(figsize=(10, 4))
colors = [ACCENT if i == monthly_avg.idxmax() else PALETTE for i in monthly_avg.index]
ax.bar(monthly_avg.index, monthly_avg.values, color=colors, width=0.7, edgecolor='white')
ax.set_title('Average Daily Items Sold by Month (across all years)')
ax.set_xlabel('Month')
ax.set_ylabel('Avg Items Sold')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_labels)
plt.tight_layout()
plt.savefig('eda_outputs/04_monthly_avg_sales.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.5 Year-over-year comparison

In [ ]:
daily_totals3 = df_open.groupby(df_open['Date'].dt.date)['total_items_sold'].sum().reset_index()
daily_totals3.columns = ['date', 'total']
daily_totals3['year']  = pd.to_datetime(daily_totals3['date']).dt.year
daily_totals3['month'] = pd.to_datetime(daily_totals3['date']).dt.month

yoy = daily_totals3.groupby(['year', 'month'])['total'].mean().unstack(level=0)

fig, ax = plt.subplots(figsize=(12, 4))
year_colors = ['#2E4057', '#E07A5F', '#3D9970', '#8B4513']
for col, color in zip(yoy.columns, year_colors):
    ax.plot(yoy.index, yoy[col], marker='o', markersize=4,
            label=str(col), color=color, linewidth=2)
ax.set_title('Year-over-Year: Average Daily Items Sold by Month')
ax.set_xlabel('Month')
ax.set_ylabel('Avg Daily Items Sold')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_labels)
ax.legend(title='Year')
plt.tight_layout()
plt.savefig('eda_outputs/05_yoy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Product & Category Breakdown
# ==========================================

### 4.1 Top 20 best-selling SKUs

In [ ]:
sku_totals = df[sku_cols].sum().sort_values(ascending=False)
top20 = sku_totals.head(20)

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(top20.index[::-1], top20.values[::-1],
               color=[ACCENT if i == 0 else PALETTE for i in range(19, -1, -1)],
               edgecolor='white')
ax.set_title('Top 20 Best-Selling SKUs (total units, 2022–2025)')
ax.set_xlabel('Total Units Sold')
for bar, val in zip(bars, top20.values[::-1]):
    ax.text(bar.get_width() + sku_totals.max() * 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:,.0f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('eda_outputs/06_top20_skus.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.2 Category-level breakdown

In [ ]:
# ── Define categories matching your preprocessing groupings ─────────────────
# Adjust these lists to match exactly what survived into your final CSV
categories = {
    'Hot Drinks':    [c for c in sku_cols if any(x in c for x in
                      ['Latte', 'Cappuc', 'HotC', 'Mocha', 'Amer', 'Espresso',
                       'Tea', 'Chai', 'Flat White', 'Macchiato', 'Babyccino'])],
    'Cold Drinks':   [c for c in sku_cols if any(x in c for x in
                      ['Juice', 'Smoothie', 'Splash', 'Milkshake', 'Iced',
                       'Lemonade', 'Frappé', 'Frappe', 'Cold Brew', 'Shake'])],
    'Breakfast':     [c for c in sku_cols if any(x in c for x in
                      ['Toast', 'Egg', 'Bacon', 'Sausage', 'Porridge',
                       'Waffle', 'Pancake', 'Brioche', 'Sourdough', 'Multiseed', 'Brown Toast'])],
    'Mains/Lunch':   [c for c in sku_cols if any(x in c for x in
                      ['Burger', 'Wrap', 'Sandwich', 'Toastie', 'Focaccia',
                       'BLT', 'Chicken', 'Beef', 'Soup', 'Macaroni', 'Mac and Cheese'])],
    'Snacks/Sides':  [c for c in sku_cols if any(x in c for x in
                      ['Chips', 'Cake', 'Brownie', 'Muffin', 'Scone', 'Flapjack',
                       'Biscuit', 'Bar', 'Loaf', 'Cookie', 'Pastry', 'Croissant'])],
}

category_totals = {cat: df[cols].sum().sum() for cat, cols in categories.items() if cols}
category_totals = pd.Series(category_totals).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
axes[0].bar(category_totals.index, category_totals.values,
            color=[PALETTE, ACCENT, '#3D9970', '#8B4513', '#6B4F8A'][:len(category_totals)],
            edgecolor='white')
axes[0].set_title('Total Units Sold by Category')
axes[0].set_ylabel('Total Units')
axes[0].tick_params(axis='x', rotation=20)

# Pie chart
axes[1].pie(category_totals.values, labels=category_totals.index,
            autopct='%1.1f%%',
            colors=[PALETTE, ACCENT, '#3D9970', '#8B4513', '#6B4F8A'][:len(category_totals)],
            startangle=90)
axes[1].set_title('Category Share of Total Sales')

plt.suptitle('Sales by Product Category (2022–2025)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_outputs/07_category_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()
print(category_totals)

### 4.3 Top SKU trends over time (are some SKUs growing or dying?)

In [ ]:
# Take top 6 SKUs and plot their monthly trend
top6_skus = sku_totals.head(6).index.tolist()

df_monthly = df.copy()
df_monthly['YearMonth'] = df_monthly['Date'].dt.to_period('M')
monthly_sku = df_monthly.groupby('YearMonth')[top6_skus].sum()
monthly_sku.index = monthly_sku.index.to_timestamp()

fig, ax = plt.subplots(figsize=(14, 5))
colors6 = ['#2E4057','#E07A5F','#3D9970','#8B4513','#6B4F8A','#C0392B']
for sku, color in zip(top6_skus, colors6):
    ax.plot(monthly_sku.index, monthly_sku[sku], label=sku, color=color, linewidth=1.5)
ax.set_title('Monthly Sales Trend — Top 6 SKUs')
ax.set_xlabel('Date')
ax.set_ylabel('Units Sold')
ax.legend(fontsize=9, ncol=2)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('eda_outputs/08_top6_sku_trends.png', dpi=150, bbox_inches='tight')
plt.show()

# Weather Sensitivity Analysis
# ==========================================

### 5.1 Correlation heatmap — weather vs total sales

In [ ]:
weather_cols = [
    'apparent_temperature', 'precipitation', 'snowfall',
    'relative_humidity_2m', 'cloud_cover', 'visibility',
    'wind_speed_10m', 'wind_gusts_10m'
]
corr_df = df[weather_cols + ['total_items_sold']].dropna()
corr_matrix = corr_df.corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.zeros_like(corr_matrix, dtype=bool)
mask[np.triu_indices_from(mask, k=1)] = True
sns.heatmap(
    corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, square=True, linewidths=0.5,
    cbar_kws={'shrink': 0.8}, ax=ax
)
ax.set_title('Pearson Correlation: Weather Variables vs Total Sales')
plt.tight_layout()
plt.savefig('eda_outputs/09_weather_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nCorrelations with total_items_sold:')
print(corr_matrix['total_items_sold'].drop('total_items_sold').sort_values(key=abs, ascending=False))

### 5.2 Temperature vs sales scatter

In [ ]:
daily_weather = df_open.groupby(df_open['Date'].dt.date).agg(
    total_sold=('total_items_sold', 'sum'),
    avg_temp=('apparent_temperature', 'mean'),
    avg_rain=('precipitation', 'mean'),
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Temperature
axes[0].scatter(daily_weather['avg_temp'], daily_weather['total_sold'],
                alpha=0.3, s=15, color=PALETTE)
# Trend line
z = np.polyfit(daily_weather['avg_temp'].dropna(),
               daily_weather.loc[daily_weather['avg_temp'].notna(), 'total_sold'], 1)
p = np.poly1d(z)
x_line = np.linspace(daily_weather['avg_temp'].min(), daily_weather['avg_temp'].max(), 100)
axes[0].plot(x_line, p(x_line), color=ACCENT, linewidth=2, label='Trend')
axes[0].set_title('Apparent Temperature vs Daily Sales')
axes[0].set_xlabel('Apparent Temperature (°C)')
axes[0].set_ylabel('Total Items Sold')
axes[0].legend()

# Precipitation
axes[1].scatter(daily_weather['avg_rain'], daily_weather['total_sold'],
                alpha=0.3, s=15, color=PALETTE)
z2 = np.polyfit(daily_weather['avg_rain'].dropna(),
                daily_weather.loc[daily_weather['avg_rain'].notna(), 'total_sold'], 1)
p2 = np.poly1d(z2)
x_line2 = np.linspace(0, daily_weather['avg_rain'].max(), 100)
axes[1].plot(x_line2, p2(x_line2), color=ACCENT, linewidth=2, label='Trend')
axes[1].set_title('Precipitation vs Daily Sales')
axes[1].set_xlabel('Avg Precipitation (mm)')
axes[1].set_ylabel('Total Items Sold')
axes[1].legend()

plt.tight_layout()
plt.savefig('eda_outputs/10_weather_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.3 Sales distribution by weather condition

In [ ]:
# WMO weather code groups: 0=Clear, 1-3=Cloudy, 51-67=Drizzle/Rain, 71-77=Snow, 80-99=Showers/Storm
def classify_weather(code):
    if pd.isna(code):    return 'Unknown'
    code = int(code)
    if code == 0:        return 'Clear'
    if code <= 3:        return 'Cloudy'
    if code <= 49:       return 'Foggy'
    if code <= 69:       return 'Rain/Drizzle'
    if code <= 79:       return 'Snow'
    return 'Showers/Storm'

df_open['weather_type'] = df_open['weather_code'].apply(classify_weather)
weather_sales = df_open.groupby('weather_type')['total_items_sold'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(weather_sales.index, weather_sales.values, color=PALETTE, edgecolor='white')
ax.set_title('Average Hourly Sales by Weather Condition')
ax.set_xlabel('Weather Type')
ax.set_ylabel('Avg Items Sold per Hour')
plt.tight_layout()
plt.savefig('eda_outputs/11_weather_condition_sales.png', dpi=150, bbox_inches='tight')
plt.show()
print(weather_sales)

# External Factors: Events & Holidays
# ==========================================

### 6.1 Holiday vs non-holiday sales

In [ ]:
holiday_comparison = df_open.groupby('is_holiday')['total_items_sold'].agg(['mean', 'median', 'std'])
if len(holiday_comparison) == 2:
    holiday_comparison.index = ['Non-Holiday', 'Holiday']
elif len(holiday_comparison) == 1:
    idx_val = holiday_comparison.index[0]
    holiday_comparison.index = ['Holiday'] if idx_val == 1 else ['Non-Holiday']
print('=== Holiday vs Non-Holiday (hourly) ===')
print(holiday_comparison.round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar comparison
if len(holiday_comparison) == 2:
    axes[0].bar(holiday_comparison.index, holiday_comparison['mean'],
                color=[PALETTE, ACCENT], edgecolor='white', width=0.5)
else:
    axes[0].bar(holiday_comparison.index, holiday_comparison['mean'],
                color=PALETTE, edgecolor='white', width=0.5)
axes[0].set_title('Average Hourly Sales: Holiday vs Non-Holiday')
axes[0].set_ylabel('Avg Items Sold')

# Box plot
box_data = []
labels = []
if 0 in df_open['is_holiday'].values:
    box_data.append(df_open[df_open['is_holiday'] == 0]['total_items_sold'].values)
    labels.append('Non-Holiday')
if 1 in df_open['is_holiday'].values:
    box_data.append(df_open[df_open['is_holiday'] == 1]['total_items_sold'].values)
    labels.append('Holiday')

if box_data:
    bp = axes[1].boxplot(box_data, patch_artist=True, labels=labels)
    if len(bp['boxes']) > 0: bp['boxes'][0].set_facecolor(PALETTE)
    if len(bp['boxes']) > 1: bp['boxes'][1].set_facecolor(ACCENT)
    for element in ['whiskers', 'caps', 'medians']:
        for item in bp[element]:
            item.set_color('black')
else:
    axes[1].text(0.5, 0.5, 'No data available', ha='center', va='center')
axes[1].set_title('Sales Distribution: Holiday vs Non-Holiday')
axes[1].set_ylabel('Items Sold')

plt.tight_layout()
plt.savefig('eda_outputs/12_holiday_impact.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.2 Impact by holiday importance score

In [ ]:
importance_sales = df_open.groupby('holiday_importance')['total_items_sold'].mean()

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(importance_sales.index.astype(str), importance_sales.values,
       color=PALETTE, edgecolor='white', width=0.6)
ax.set_title('Avg Hourly Sales by Holiday Importance Score (0 = no holiday)')
ax.set_xlabel('Holiday Importance Score')
ax.set_ylabel('Avg Items Sold')
plt.tight_layout()
plt.savefig('eda_outputs/13_holiday_importance_sales.png', dpi=150, bbox_inches='tight')
plt.show()
print(importance_sales)

### 6.3 Festival & match day impact

In [ ]:
# Combined event impact summary
event_summary = pd.DataFrame({
    'Group': ['No Event', 'Festival Day', 'Match Day', 'Both'],
    'Avg Sales': [
        df_open[(df_open['is_festival']==0) & (df_open['is_match_day']==0)]['total_items_sold'].mean(),
        df_open[(df_open['is_festival']==1) & (df_open['is_match_day']==0)]['total_items_sold'].mean(),
        df_open[(df_open['is_festival']==0) & (df_open['is_match_day']==1)]['total_items_sold'].mean(),
        df_open[(df_open['is_festival']==1) & (df_open['is_match_day']==1)]['total_items_sold'].mean(),
    ]
}).dropna()

fig, ax = plt.subplots(figsize=(8, 4))
colors_ev = [PALETTE, ACCENT, '#3D9970', '#8B4513']
ax.bar(event_summary['Group'], event_summary['Avg Sales'],
       color=colors_ev[:len(event_summary)], edgecolor='white', width=0.5)
ax.set_title('Average Hourly Sales by Event Type')
ax.set_ylabel('Avg Items Sold')
plt.tight_layout()
plt.savefig('eda_outputs/14_event_impact.png', dpi=150, bbox_inches='tight')
plt.show()
print(event_summary.set_index('Group'))

### 6.4 Days Until Next Holiday — countdown effect

In [ ]:
# Does demand change in the run-up to a holiday?
countdown = df_open[df_open['Days_Until_Next_Holiday'] <= 14].groupby(
    'Days_Until_Next_Holiday'
)['total_items_sold'].mean().sort_index()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(countdown.index, countdown.values, color=PALETTE, edgecolor='white', width=0.7)
ax.axvline(x=0, color=ACCENT, linestyle='--', linewidth=1.5, label='Holiday Day')
ax.set_title('Average Hourly Sales by Days Until Next Holiday (0–14 days)')
ax.set_xlabel('Days Until Next Holiday (0 = holiday itself)')
ax.set_ylabel('Avg Items Sold')
ax.legend()
plt.tight_layout()
plt.savefig('eda_outputs/15_countdown_effect.png', dpi=150, bbox_inches='tight')
plt.show()

# Anomaly Detection
# ==========================================

In [ ]:
# ── Daily totals distribution ─────────────────────────────────────────────────
daily_all = df.groupby(df['Date'].dt.date)['total_items_sold'].sum()

Q1 = daily_all.quantile(0.25)
Q3 = daily_all.quantile(0.75)
IQR = Q3 - Q1
lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

outliers_high = daily_all[daily_all > upper_fence]
outliers_low  = daily_all[(daily_all < lower_fence) & (daily_all > 0)]  # exclude full closures

print(f'IQR bounds  : {lower_fence:.0f} – {upper_fence:.0f}')
print(f'High outlier days : {len(outliers_high)}')
print(f'Low outlier days  : {len(outliers_low)}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histogram
axes[0].hist(daily_all.values, bins=60, color=PALETTE, edgecolor='white', alpha=0.85)
axes[0].axvline(upper_fence, color=ACCENT, linestyle='--', linewidth=1.5, label=f'Upper fence ({upper_fence:.0f})')
axes[0].axvline(lower_fence, color='orange', linestyle='--', linewidth=1.5, label=f'Lower fence ({lower_fence:.0f})')
axes[0].set_title('Distribution of Daily Total Items Sold')
axes[0].set_xlabel('Items Sold')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Time series with outliers flagged
axes[1].plot(daily_all.index, daily_all.values, color=PALETTE, linewidth=0.8, alpha=0.7)
if len(outliers_high) > 0:
    axes[1].scatter(outliers_high.index, outliers_high.values,
                    color=ACCENT, zorder=5, s=30, label='High outliers')
if len(outliers_low) > 0:
    axes[1].scatter(outliers_low.index, outliers_low.values,
                    color='orange', zorder=5, s=30, label='Low outliers')
axes[1].set_title('Daily Sales with Outliers Flagged')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Items Sold')
axes[1].legend()
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.savefig('eda_outputs/16_outlier_detection.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 5 highest-sales days:')
print(daily_all.sort_values(ascending=False).head())
print('\nTop 5 lowest-sales days (excluding full closures):')
print(outliers_low.sort_values().head())

# Summary of Findings
# ==========================================

In [ ]:
# ── Auto-generate key stats for your write-up ────────────────────────────────
peak_hour     = hourly_avg.idxmax()
busiest_day   = day_labels[weekday_avg.idxmax()]
busiest_month = month_labels[monthly_avg.idxmax() - 1]
top_sku       = sku_totals.index[0]

holiday_lift = (
    (df_open[df_open['is_holiday']==1]['total_items_sold'].mean() /
     df_open[df_open['is_holiday']==0]['total_items_sold'].mean() - 1) * 100
)
festival_lift = (
    (df_open[df_open['is_festival']==1]['total_items_sold'].mean() /
     df_open[df_open['is_festival']==0]['total_items_sold'].mean() - 1) * 100
)
match_lift = (
    (df_open[df_open['is_match_day']==1]['total_items_sold'].mean() /
     df_open[df_open['is_match_day']==0]['total_items_sold'].mean() - 1) * 100
)
temp_corr = corr_df.corr()['total_items_sold']['apparent_temperature']

print('=' * 55)
print('       EDA KEY FINDINGS SUMMARY')
print('=' * 55)
print(f'Dataset            : {len(df):,} rows | {len(sku_cols)} SKUs')
print(f'Date range         : {df["Date"].min().date()} → {df["Date"].max().date()}')
print(f'Closure days       : {len(full_closure_days)}')
print(f'High outlier days  : {len(outliers_high)}')
print()
print(f'Peak hour          : {peak_hour:02d}:00')
print(f'Busiest weekday    : {busiest_day}')
print(f'Busiest month      : {busiest_month}')
print(f'Top SKU            : {top_sku} ({sku_totals.iloc[0]:,.0f} units)')
print()
print(f'Holiday lift       : {holiday_lift:+.1f}% vs non-holiday')
print(f'Festival lift      : {festival_lift:+.1f}% vs non-festival')
print(f'Match day lift     : {match_lift:+.1f}% vs non-match')
print(f'Temp correlation   : {temp_corr:+.3f} (with total sales)')
print('=' * 55)